In [0]:
# =============================================================================
# TRAVEL BOOKING SCD2 MERGE PROJECT - DATA QUALITY: BOOKING DATA VALIDATION
# =============================================================================
# This notebook performs comprehensive data quality checks on booking data
# Purpose: Validates booking data integrity using PySpark (optimized, no PyDeequ)
# Data Quality: Checks completeness, non-negativity, and business rules
# Output: Logs DQ results and raises exceptions for failed validations

from pyspark.sql import functions as F

# =============================================================================
# PARAMETER EXTRACTION WITH DEFAULTS
# =============================================================================

import datetime as _dt
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "dbx-external-catalog"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

# =============================================================================
# DQ RESULTS STORAGE SETUP
# =============================================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.ops")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{catalog}`.ops.dq_results (
  business_date DATE,
  dataset STRING,
  check_name STRING,
  status STRING,
  constraint STRING,
  message STRING,
  recorded_at TIMESTAMP
) USING DELTA
""")

# =============================================================================
# SOURCE DATA PREPARATION
# =============================================================================

src = spark.table(f"`{catalog}`.bronze.booking_inc") \
           .where(F.col("business_date") == F.to_date(F.lit(arrival_date)))

# =============================================================================
# DATA QUALITY CHECKS IMPLEMENTATION (OPTIMIZED - SINGLE PASS)
# =============================================================================

from pyspark.sql import Row

# ---- Optional column pruning (performance boost if wide table) ----
src = src.select("customer_id", "amount", "quantity", "discount")

# ---- Single pass aggregation ----
agg_df = src.agg(
    F.count("*").alias("row_count"),

    # Completeness checks
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_id"),
    F.sum(F.when(F.col("amount").isNull(), 1).otherwise(0)).alias("null_amount"),

    # Non-negative checks
    F.sum(F.when(F.col("amount") < 0, 1).otherwise(0)).alias("neg_amount"),
    F.sum(F.when(F.col("quantity") < 0, 1).otherwise(0)).alias("neg_quantity"),
    F.sum(F.when(F.col("discount") < 0, 1).otherwise(0)).alias("neg_discount")
)

metrics = agg_df.collect()[0]

dq_results = []

# ---- Check 1: hasSize ----
if metrics["row_count"] > 0:
    dq_results.append(Row("Booking Data Check","Success","hasSize > 0","Success",f"Row count = {metrics['row_count']}"))
else:
    dq_results.append(Row("Booking Data Check","Error","hasSize > 0","Failure","No data found"))

# ---- Check 2: customer_id completeness ----
if metrics["null_customer_id"] == 0:
    dq_results.append(Row("Booking Data Check","Success","customer_id NOT NULL","Success","No nulls found"))
else:
    dq_results.append(Row("Booking Data Check","Error","customer_id NOT NULL","Failure",f"{metrics['null_customer_id']} nulls"))

# ---- Check 3: amount completeness ----
if metrics["null_amount"] == 0:
    dq_results.append(Row("Booking Data Check","Success","amount NOT NULL","Success","No nulls found"))
else:
    dq_results.append(Row("Booking Data Check","Error","amount NOT NULL","Failure",f"{metrics['null_amount']} nulls"))

# ---- Check 4: amount non-negative ----
if metrics["neg_amount"] == 0:
    dq_results.append(Row("Booking Data Check","Success","amount >= 0","Success","No negative values"))
else:
    dq_results.append(Row("Booking Data Check","Error","amount >= 0","Failure",f"{metrics['neg_amount']} negative values"))

# ---- Check 5: quantity non-negative ----
if metrics["neg_quantity"] == 0:
    dq_results.append(Row("Booking Data Check","Success","quantity >= 0","Success","No negative values"))
else:
    dq_results.append(Row("Booking Data Check","Error","quantity >= 0","Failure",f"{metrics['neg_quantity']} negative values"))

# ---- Check 6: discount non-negative ----
if metrics["neg_discount"] == 0:
    dq_results.append(Row("Booking Data Check","Success","discount >= 0","Success","No negative values"))
else:
    dq_results.append(Row("Booking Data Check","Error","discount >= 0","Failure",f"{metrics['neg_discount']} negative values"))

# ---- Convert to DataFrame ----
df = spark.createDataFrame(
    dq_results,
    ["check", "check_status", "constraint", "constraint_status", "constraint_message"]
)

display(df)

# ---- Overall result ----
result_status = "Success" if all(r[3] == "Success" for r in dq_results) else "Error"

# =============================================================================
# DQ RESULTS LOGGING
# =============================================================================

out = (df
  .withColumn("business_date", F.to_date(F.lit(arrival_date)))
  .withColumn("dataset", F.lit("booking_inc"))
  .withColumn("recorded_at", F.current_timestamp()))

out.select(
    "business_date",
    "dataset",
    F.col("check").alias("check_name"),
    F.col("constraint_status").alias("status"),
    "constraint",
    F.col("constraint_message").alias("message"),
    "recorded_at"
).write.mode("append").option("mergeSchema", "true") \
 .saveAsTable(f"`{catalog}`.ops.dq_results")

# =============================================================================
# DQ VALIDATION AND ERROR HANDLING
# =============================================================================

if result_status != "Success":
    raise ValueError("DQ failed for bookings")

print("Booking DQ passed")